In [4]:
# %% [markdown]
# # Step 8: Test Complete Application
# ## Verify all components are working

# %%
import os
import sys
import importlib
import numpy as np
import traceback

# %%
print("="*50)
print("🔍 TESTING NITROSENSE AI APPLICATION")
print("="*50)

# Test 1: Check folder structure
print("\n📁 Test 1: Checking folder structure...")
required_folders = ['models', 'utils', 'templates', 'static', 'u2net', 'u2net/weights', 'u2net/model', 'static/uploads']
all_folders_ok = True

for folder in required_folders:
    if os.path.exists(folder):
        print(f"  ✅ {folder} folder exists")
    else:
        print(f"  ❌ {folder} folder missing")
        all_folders_ok = False

# %%
# Test 2: Check required files
print("\n📄 Test 2: Checking required files...")

required_files = [
    ('models', 'Pyramid_fusion_densenet121_model.h5'),
    ('utils', 'background_removal.py'),
    ('utils', 'edge_detection.py'),
    ('utils', 'preprocessing.py'),
    ('utils/scalers', 'scaler_y.pkl'),
    ('utils/scalers', 'continuous_scaler.pkl'),
    ('utils/scalers', 'days_scaler.pkl'),
    ('utils/scalers', 'nitrogen_map.pkl'),
    ('utils', 'temperature_means.pkl'),
    ('templates', 'index.html'),
    ('templates', 'estimation.html'),
    ('templates', 'about.html'),
    ('templates', 'team.html'),
    ('templates', 'how-it-works.html'),
    ('templates', 'contact.html'),
    ('templates', 'capture.html'),
    ('templates', 'result.html'),
    ('u2net/weights', 'u2net.pth'),
]

all_files_ok = True
for folder, filename in required_files:
    filepath = os.path.join(folder, filename)
    if os.path.exists(filepath):
        file_size = os.path.getsize(filepath) / (1024 * 1024) if filename.endswith('.pth') or filename.endswith('.h5') else 0
        size_display = f" ({file_size:.2f} MB)" if file_size > 0 else ""
        print(f"  ✅ {filepath} exists{size_display}")
    else:
        print(f"  ❌ {filepath} missing")
        all_files_ok = False

# %%
# Test 3: Import utilities
print("\n🔧 Test 3: Testing utility imports...")

sys.path.append(os.path.join(os.getcwd(), 'utils'))

try:
    from background_removal import BackgroundRemover
    print("  ✅ BackgroundRemover imported")
except Exception as e:
    print(f"  ❌ BackgroundRemover import failed: {e}")

try:
    from edge_detection import EdgeDetector
    print("  ✅ EdgeDetector imported")
except Exception as e:
    print(f"  ❌ EdgeDetector import failed: {e}")

try:
    from preprocessing import ImagePreprocessor
    print("  ✅ ImagePreprocessor imported")
except Exception as e:
    print(f"  ❌ ImagePreprocessor import failed: {e}")

# %%
# Test 4: Test BackgroundRemover initialization
print("\n🌄 Test 4: Testing BackgroundRemover initialization...")

try:
    # Test with portable initialization (no path needed)
    bg_remover = BackgroundRemover()
    print("  ✅ BackgroundRemover initialized successfully")
    
    # Test with a dummy image
    dummy_image = np.zeros((100, 100, 3), dtype=np.uint8)
    dummy_image[:] = [100, 150, 200]
    
    result, mask = bg_remover.remove_background(dummy_image, target_size=(224, 224))
    print(f"  ✅ Background removal works - output shape: {result.shape}")
    if mask is not None:
        print(f"  ✅ Mask generated - shape: {mask.shape}")
    else:
        print(f"  ℹ️ No mask generated (running in fallback mode)")
    
except Exception as e:
    print(f"  ❌ BackgroundRemover test failed: {e}")
    traceback.print_exc()

# %%
# Test 5: Quick test of preprocessing pipeline
print("\n🖼️ Test 5: Testing preprocessing pipeline...")

try:
    # Create a dummy image
    import cv2
    from PIL import Image
    
    dummy_image = np.zeros((100, 100, 3), dtype=np.uint8)
    dummy_image[:] = [100, 150, 200]
    
    # Test edge detection
    edge_detector = EdgeDetector()
    edge_image, edges = edge_detector.detect_edges(dummy_image)
    print(f"  ✅ Edge detection works - output shape: {edge_image.shape}")
    
    # Test preprocessor - use 224x224 to match model
    preprocessor = ImagePreprocessor(target_size=(224, 224))
    processed = preprocessor.preprocess_for_model(edge_image)
    print(f"  ✅ Preprocessor works with target_size=(224, 224) - output shape: {processed.shape}")
    print(f"  ✅ Output range: [{processed.min():.2f}, {processed.max():.2f}]")
    
except Exception as e:
    print(f"  ❌ Preprocessing test failed: {e}")
    traceback.print_exc()

# %%
# Test 6: Check model file
print("\n🤖 Test 6: Checking model file...")

model_path = os.path.join('models', 'Pyramid_fusion_densenet121_model.h5')
if os.path.exists(model_path):
    file_size = os.path.getsize(model_path) / (1024 * 1024)  # MB
    print(f"  ✅ Model file found: {model_path}")
    print(f"  📊 File size: {file_size:.2f} MB")
    
    # Quick model structure check without loading
    try:
        import h5py
        with h5py.File(model_path, 'r') as f:
            print(f"  ✅ Model file is valid HDF5 format")
    except Exception as e:
        print(f"  ⚠️ Could not verify model file: {e}")
else:
    print(f"  ❌ Model file not found at: {model_path}")

# %%
# Test 7: Check scaler files
print("\n📊 Test 7: Checking scaler files...")

try:
    import pickle
    
    scaler_files = [
        ('utils/scalers/scaler_y.pkl', 'Target scaler'),
        ('utils/scalers/continuous_scaler.pkl', 'Temperature scaler'),
        ('utils/scalers/days_scaler.pkl', 'Days scaler'),
        ('utils/scalers/nitrogen_map.pkl', 'Nitrogen map'),
        ('utils/temperature_means.pkl', 'Temperature means')
    ]
    
    for filepath, description in scaler_files:
        if os.path.exists(filepath):
            with open(filepath, 'rb') as f:
                data = pickle.load(f)
            print(f"  ✅ {description} loaded successfully")
        else:
            print(f"  ❌ {filepath} missing")
            
except Exception as e:
    print(f"  ❌ Scaler test failed: {e}")
    traceback.print_exc()

# %%
# Test 8: Check Flask app syntax
print("\n⚙️ Test 8: Checking Flask app syntax...")

try:
    import py_compile
    py_compile.compile('app.py')
    print("  ✅ Flask app syntax is valid")
except Exception as e:
    print(f"  ❌ Flask app syntax error: {e}")
    traceback.print_exc()

# %%
# Test 9: Check template files for Jinja syntax
print("\n📝 Test 9: Checking template files...")

template_files = ['index.html', 'estimation.html', 'about.html', 'team.html', 
                  'how-it-works.html', 'contact.html', 'capture.html', 'result.html']

for template in template_files:
    template_path = os.path.join('templates', template)
    if os.path.exists(template_path):
        with open(template_path, 'r', encoding='utf-8') as f:
            content = f.read()
        # Quick check for Jinja syntax
        if '{{' in content and '}}' in content:
            print(f"  ✅ {template} contains Jinja syntax")
        else:
            print(f"  ℹ️ {template} appears to be a static page (no Jinja syntax)")
    else:
        print(f"  ❌ {template} missing")

# %%
# Test 10: Check U2Net weights
print("\n🔧 Test 10: Checking U2Net weights...")

weights_path = os.path.join('u2net', 'weights', 'u2net.pth')
if os.path.exists(weights_path):
    file_size = os.path.getsize(weights_path) / (1024 * 1024)  # MB
    print(f"  ✅ U2Net weights found: {weights_path}")
    print(f"  📊 File size: {file_size:.2f} MB")
else:
    print(f"  ⚠️ U2Net weights not found at: {weights_path}")
    print("     Background removal will run in fallback mode")

# %%
# Summary
print("\n" + "="*50)
print("📊 TEST SUMMARY")
print("="*50)

if all_folders_ok and all_files_ok:
    print("\n✅ All folders and files are in place!")
    print("\n🎉 Your NitroSense AI application is ready to run!")
else:
    print("\n⚠️ Some files/folders are missing. Check the ❌ marks above.")

# Check if any critical components are missing
critical_missing = []
if not os.path.exists('models/Pyramid_fusion_densenet121_model.h5'):
    critical_missing.append("Model file")
if not os.path.exists('utils/background_removal.py'):
    critical_missing.append("Background remover")
if not os.path.exists('templates/index.html'):
    critical_missing.append("Templates")

if critical_missing:
    print("\n❌ Critical components missing:")
    for item in critical_missing:
        print(f"   - {item}")
    print("\nPlease fix these before running the application.")
else:
    print("\n✅ All critical components present!")

print("\n" + "="*50)
print("🚀 TO RUN THE APPLICATION:")
print("="*50)
print("\nOpen a terminal/command prompt and run:")
print("  cd C:\\Users\\HP\\NitroSense-AI")
print("  python app.py")
print("\nThen open your browser and go to:")
print("  http://localhost:5000")

# Optional: Print network URL for sharing
import socket
hostname = socket.gethostname()
try:
    local_ip = socket.gethostbyname(hostname)
    print(f"  http://{local_ip}:5000 (for other devices on same network)")
except:
    print("  Could not determine local IP")

print("\n" + "="*50)


# %% [markdown]
# # Additional Test: Test Model Prediction with Dummy Data

# %%
print("\n" + "="*50)
print("🧪 TESTING MODEL PREDICTION WITH DUMMY DATA")
print("="*50)

try:
    import tensorflow as tf
    from tensorflow.keras import backend as K
    import pickle
    
    # ============= DEFINE CUSTOM ACTIVATION FUNCTIONS =============
    def mish_activation(x):
        """
        Mish activation function: x * tanh(softplus(x))
        """
        return x * tf.math.tanh(tf.math.softplus(x))
    
    def swish_activation(x):
        """
        Swish activation function: x * sigmoid(x)
        """
        return x * tf.nn.sigmoid(x)
    # ============================================================
    
    # Load the model
    model_path = os.path.join('models', 'Pyramid_fusion_densenet121_model.h5')
    
    # Define custom metrics
    def rmse(y_true, y_pred):
        return K.sqrt(K.mean(K.square(y_pred - y_true)))
    
    def r2(y_true, y_pred):
        SS_res = K.sum(K.square(y_true - y_pred))
        SS_tot = K.sum(K.square(y_true - K.mean(y_true)))
        return 1 - SS_res/(SS_tot + K.epsilon())
    
    def mae(y_true, y_pred):
        return K.mean(K.abs(y_pred - y_true))
    
    # Add custom activation functions to custom_objects
    custom_objects = {
        'mse': tf.keras.losses.MeanSquaredError(),
        'MSE': tf.keras.losses.MeanSquaredError(),
        'mean_squared_error': tf.keras.losses.MeanSquaredError(),
        'mae': mae,
        'MAE': mae,
        'mean_absolute_error': mae,
        'rmse': rmse,
        'RMSE': rmse,
        'root_mean_squared_error': rmse,
        'r2': r2,
        'R2': r2,
        'r_squared': r2,
        'R_squared': r2,
        # Add custom activation functions
        'mish_activation': mish_activation,
        'swish_activation': swish_activation,
    }
    
    # Register custom activations globally (as fallback)
    tf.keras.utils.get_custom_objects()['mish_activation'] = mish_activation
    tf.keras.utils.get_custom_objects()['swish_activation'] = swish_activation
    
    # Load model
    print("Loading model...")
    model = tf.keras.models.load_model(model_path, custom_objects=custom_objects, compile=False)
    print("✅ Model loaded successfully")
    
    # Print model input shapes to verify
    print("\n📊 Model input shapes:")
    for i, input_layer in enumerate(model.inputs):
        print(f"   Input {i+1}: {input_layer.name} - Shape: {input_layer.shape}")
    
    # Create dummy test data with CORRECT size (224x224 for DenseNet121)
    print("\n🔄 Creating dummy test data with size 224x224...")
    dummy_image = np.random.rand(1, 224, 224, 3).astype(np.float32)
    dummy_features = np.random.rand(1, 4).astype(np.float32)
    
    print(f"   Dummy image shape: {dummy_image.shape}")
    print(f"   Dummy features shape: {dummy_features.shape}")
    print(f"   Dummy features: {dummy_features[0]}")
    
    # Make prediction
    print("\n🔄 Making prediction...")
    prediction = model.predict([dummy_image, dummy_features], verbose=0)
    print(f"✅ Model prediction successful: {prediction[0][0]:.6f}")
    print(f"   Prediction shape: {prediction.shape}")
    
    # Test with different feature values to see if output changes
    print("\n🔄 Testing with different feature values:")
    
    test_features = [
        np.array([[0.1, 0.1, 0, 0.1]], dtype=np.float32),
        np.array([[0.5, 0.5, 3, 0.5]], dtype=np.float32),
        np.array([[0.9, 0.9, 7, 0.9]], dtype=np.float32)
    ]
    
    for i, features in enumerate(test_features):
        pred = model.predict([dummy_image, features], verbose=0)
        print(f"   Test {i+1} - Features: {features[0]} → Prediction: {pred[0][0]:.6f}")
    
    # Test with multiple feature variations to check if model responds to inputs
    print("\n🔍 Testing prediction variation with different features:")
    feature_sets = [
        np.array([[0.0, 0.0, 0, 0.0]], dtype=np.float32),
        np.array([[0.2, 0.2, 1, 0.2]], dtype=np.float32),
        np.array([[0.4, 0.4, 3, 0.4]], dtype=np.float32),
        np.array([[0.6, 0.6, 5, 0.6]], dtype=np.float32),
        np.array([[0.8, 0.8, 6, 0.8]], dtype=np.float32),
        np.array([[1.0, 1.0, 7, 1.0]], dtype=np.float32)
    ]
    
    predictions = []
    for features in feature_sets:
        pred = model.predict([dummy_image, features], verbose=0)[0][0]
        predictions.append(pred)
        print(f"   Features {features[0]} → {pred:.6f}")
    
    # Check if all predictions are the same
    if len(set(predictions)) == 1:
        print("\n⚠️ WARNING: All predictions are identical! Model might not be working correctly.")
    else:
        print("\n✅ Model produces varying outputs for different inputs - good!")
    
    # Test with scaler_y if available
    try:
        scaler_y_path = os.path.join('utils/scalers', 'scaler_y.pkl')
        if os.path.exists(scaler_y_path):
            with open(scaler_y_path, 'rb') as f:
                scaler_y = pickle.load(f)
            print(f"\n📊 Testing denormalization with scaler_y:")
            print(f"   Scaler_y mean: {scaler_y.mean_[0]:.4f}, scale: {scaler_y.scale_[0]:.4f}")
            
            # Denormalize the predictions
            pred_array = np.array([[p] for p in predictions])
            original_values = scaler_y.inverse_transform(pred_array)
            print(f"\n📊 Denormalized values (should be in range 2-5%):")
            for i, (norm, orig) in enumerate(zip(predictions, original_values)):
                print(f"   Normalized: {norm:.6f} → Original Nitrogen: {orig[0]:.2f}%")
    except Exception as e:
        print(f"   Could not test denormalization: {e}")
    
    # Compare with the sample prediction from model testing
    print(f"\n📊 Note: After denormalization, values should be in the range 2-5%")
    
except Exception as e:
    print(f"❌ Model test failed: {e}")
    traceback.print_exc()

🔍 TESTING NITROSENSE AI APPLICATION

📁 Test 1: Checking folder structure...
  ✅ models folder exists
  ✅ utils folder exists
  ✅ templates folder exists
  ✅ static folder exists
  ✅ u2net folder exists
  ✅ u2net/weights folder exists
  ✅ u2net/model folder exists
  ✅ static/uploads folder exists

📄 Test 2: Checking required files...
  ✅ models\Pyramid_fusion_densenet121_model.h5 exists (59.70 MB)
  ✅ utils\background_removal.py exists
  ✅ utils\edge_detection.py exists
  ✅ utils\preprocessing.py exists
  ✅ utils/scalers\scaler_y.pkl exists
  ✅ utils/scalers\continuous_scaler.pkl exists
  ✅ utils/scalers\days_scaler.pkl exists
  ✅ utils/scalers\nitrogen_map.pkl exists
  ✅ utils\temperature_means.pkl exists
  ✅ templates\index.html exists
  ✅ templates\estimation.html exists
  ✅ templates\about.html exists
  ✅ templates\team.html exists
  ✅ templates\how-it-works.html exists
  ✅ templates\contact.html exists
  ✅ templates\capture.html exists
  ✅ templates\result.html exists
  ✅ u2net/wei

  File "app.py", line 81
    print("
          ^
SyntaxError: unterminated string literal (detected at line 81)



✅ Model loaded successfully

📊 Model input shapes:
   Input 1: image_input - Shape: (None, 224, 224, 3)
   Input 2: tabular_input - Shape: (None, 4)

🔄 Creating dummy test data with size 224x224...
   Dummy image shape: (1, 224, 224, 3)
   Dummy features shape: (1, 4)
   Dummy features: [0.7880997  0.7509771  0.22307786 0.90968657]

🔄 Making prediction...
✅ Model prediction successful: -1.375976
   Prediction shape: (1, 1)

🔄 Testing with different feature values:
   Test 1 - Features: [0.1 0.1 0.  0.1] → Prediction: -1.174216
   Test 2 - Features: [0.5 0.5 3.  0.5] → Prediction: -0.676204
   Test 3 - Features: [0.9 0.9 7.  0.9] → Prediction: -0.284522

🔍 Testing prediction variation with different features:
   Features [0. 0. 0. 0.] → -1.143696
   Features [0.2 0.2 1.  0.2] → -1.029661
   Features [0.4 0.4 3.  0.4] → -0.645686
   Features [0.6 0.6 5.  0.6] → -0.089085
   Features [0.8 0.8 6.  0.8] → -0.237164
   Features [1. 1. 7. 1.] → -0.312324

✅ Model produces varying outputs for 

In [3]:
# Diagnostic for U2Net
print("="*50)
print("🔍 U2Net DIAGNOSTIC")
print("="*50)

import os
import sys

# Check if u2net directory exists
u2net_dir = 'u2net'
model_dir = os.path.join(u2net_dir, 'model')
weights_dir = os.path.join(u2net_dir, 'weights')

print(f"\n📁 Checking directories:")
print(f"  u2net_dir exists: {os.path.exists(u2net_dir)}")
print(f"  model_dir exists: {os.path.exists(model_dir)}")
print(f"  weights_dir exists: {os.path.exists(weights_dir)}")

# Check for u2net.py
u2net_py = os.path.join(model_dir, 'u2net.py')
print(f"\n📄 Checking u2net.py:")
print(f"  Path: {u2net_py}")
print(f"  Exists: {os.path.exists(u2net_py)}")
if os.path.exists(u2net_py):
    file_size = os.path.getsize(u2net_py) / 1024
    print(f"  Size: {file_size:.2f} KB")

# Check for weights
weights_path = os.path.join(weights_dir, 'u2net.pth')
print(f"\n⚖️ Checking weights:")
print(f"  Path: {weights_path}")
print(f"  Exists: {os.path.exists(weights_path)}")
if os.path.exists(weights_path):
    file_size = os.path.getsize(weights_path) / (1024 * 1024)
    print(f"  Size: {file_size:.2f} MB")

# Try to import
print(f"\n🔄 Trying to import U2Net module:")
sys.path.insert(0, os.path.abspath(model_dir))
try:
    from u2net import U2NET
    print(f"  ✅ Successfully imported U2NET")
    print(f"  U2NET type: {type(U2NET)}")
except Exception as e:
    print(f"  ❌ Import failed: {e}")

# Initialize BackgroundRemover with verbose output
print(f"\n🔄 Initializing BackgroundRemover:")
from utils.background_removal import BackgroundRemover
bg_remover = BackgroundRemover()
print(f"  u2net_available: {bg_remover.u2net_available}")
print(f"  model loaded: {bg_remover.model is not None}")

🔍 U2Net DIAGNOSTIC

📁 Checking directories:
  u2net_dir exists: True
  model_dir exists: True
  weights_dir exists: True

📄 Checking u2net.py:
  Path: u2net\model\u2net.py
  Exists: True
  Size: 14.37 KB

⚖️ Checking weights:
  Path: u2net\weights\u2net.pth
  Exists: True
  Size: 168.12 MB

🔄 Trying to import U2Net module:
  ✅ Successfully imported U2NET
  U2NET type: <class 'type'>

🔄 Initializing BackgroundRemover:
✅ U2Net source already present
✅ BackgroundRemover initialized with U2Net model
  u2net_available: True
  model loaded: True
